In [1]:
# 1. Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the raw data (using our synthetic DataFrame from notebook 1)
df = pd.read_csv('../data/raw_data.csv')

def engineer_features(raw_df):
    """
    Transforms raw transactional data into behavioral features.
    
    ENTERPRISE NOTE: In a real production environment, features like 'geo_velocity' 
    and 'tx_count_10m' are calculated in real-time using Redis geospatial queries 
    and sliding window counters. Here, we simulate the resulting statistical distributions
    to train our models without needing months of historical user data.
    """
    df = raw_df.copy()
    n = len(df)
    np.random.seed(42)
    
    print("⚙️ Engineering Temporal Features...")
    # Extract time-based features (e.g., late-night transactions are historically riskier)
    df['hour_of_day'] = np.random.randint(0, 24, n)
    df['is_weekend'] = np.random.binomial(1, 0.28, n)
    
    # NEW: Time since last transaction (fraudsters move extremely fast)
    df['time_since_last_tx'] = np.where(df['Class'] == 1, 
                                        np.random.exponential(30, n),   # Fraud: ~30 seconds apart
                                        np.random.exponential(3600, n)) # Legit: ~1 hour apart
    
    print("⚙️ Engineering Velocity Features (Simulated)...")
    # Fraudsters usually trigger high velocity (e.g., card cloning used simultaneously across cities)
    df['geo_velocity'] = np.where(df['Class'] == 1, 
                                  np.random.exponential(150, n), # High speed (km/h)
                                  np.random.exponential(5, n))   # Normal speed (km/h)
                                  
    # Fraudsters usually test a card multiple times quickly (velocity attacks)
    df['tx_count_10m'] = np.where(df['Class'] == 1, 
                                  np.random.poisson(5, n), 
                                  np.random.poisson(1.2, n))
    
    print("⚙️ Engineering Statistical Features...")
    # Calculate Z-Score of the amount to find unusual spending spikes.
    # NOTE: In production, this is grouped by account_id. Here we simulate a global Z-score.
    df['amount_z_score'] = (df['amount'] - df['amount'].mean()) / df['amount'].std()
    
    # Safety Check: Fill any NaNs that might occur from division by zero
    df['amount_z_score'] = df['amount_z_score'].fillna(0)
    
    return df

# Apply feature engineering
engineered_df = engineer_features(df)

# View the final dataset ready for the models
print("\n✅ Feature Engineering Complete. Final Feature Set Head:")
features_to_display = [
    'amount', 'geo_velocity', 'time_since_last_tx', 
    'tx_count_10m', 'hour_of_day', 'is_weekend', 
    'amount_z_score', 'Class'
]
display(engineered_df[features_to_display].head())

# Save for the next notebook
engineered_df.to_csv('../data/engineered_data.csv', index=False)
print("\n💾 Saved engineered dataset to '../data/engineered_data.csv'")

⚙️ Engineering Temporal Features...
⚙️ Engineering Velocity Features (Simulated)...
⚙️ Engineering Statistical Features...

✅ Feature Engineering Complete. Final Feature Set Head:


,amount,geo_velocity,time_since_last_tx,tx_count_10m,hour_of_day,is_weekend,amount_z_score,Class
0,46.926809,10.500148,6803.026411,2,6,0,-0.441948,0
1,301.012143,1.866196,1836.010955,3,19,0,1.456769,0
2,131.674569,0.118039,17807.444474,0,14,0,0.191351,0
3,456.471277,111.728130,39.207369,6,10,0,2.618477,1
4,16.962487,17.678740,5594.960268,0,7,0,-0.665864,0



💾 Saved engineered dataset to '../data/engineered_data.csv'
